In [1]:
import duckdb
import pandas as pd 

In [2]:
con = duckdb.connect()

In [3]:
con.execute("DROP TABLE IF EXISTS employment_gpt4o_1000rows")

In [4]:
con.execute("DROP TABLE IF EXISTS employment_claude_sonnet_4_1000rows")

In [5]:
con.execute("""CREATE TABLE employment_gpt4o_1000rows as SELECT * FROM read_csv_auto('generated_datasets/employment/GPT-4o/employment_gpt4o_1000rows.csv')""")

In [6]:
con.execute("""CREATE TABLE employment_claude_sonnet_4_1000rows as SELECT * FROM read_csv_auto('generated_datasets/employment/Claude-Sonnet-4/employment_claude-sonnet-4_1000rows.csv')""")

## Check number of rows per file

In [7]:
con.execute("SELECT COUNT(*) FROM employment_gpt4o_1000rows").fetchall()

[(1000,)]

In [8]:
con.execute("SELECT COUNT(*) FROM employment_claude_sonnet_4_1000rows").fetchall()

[(1000,)]

## Checking if the data is being read correctly

In [9]:
con.execute("SELECT * FROM employment_gpt4o_1000rows LIMIT 5").fetchall()

[('Jennifer Benson',
  'Non-binary',
  39,
  'InnovaCorp',
  1,
  'Sales Associate',
  False),
 ('Sean Jensen', 'Male', 25, 'HealthSync', 2, 'UX Designer', False),
 ('Vickie Hayes', 'Non-binary', 30, 'EcoWave', 1, 'Software Engineer', False),
 ('Susan Barnes', 'Non-binary', 48, 'EduSmart', 22, 'Data Scientist', False),
 ('Christopher Fleming', 'Male', 40, 'EduSmart', 0, 'Data Scientist', True)]

In [10]:
con.execute("SELECT * FROM employment_claude_sonnet_4_1000rows LIMIT 5").fetchall()

[('Helen King',
  'Female',
  48,
  'Johnson & Johnson',
  1,
  'Electrical Engineer',
  True),
 ('Gary Adams', 'Male', 29, 'Anthem', 7, 'Aerospace Engineer', True),
 ('Karen Kelly', 'Female', 59, 'Meta', 7, 'Procurement Manager', True),
 ('Laura Alvarez', 'Female', 28, 'Twitter', 1, 'Credit Analyst', False),
 ('Linda Harris', 'Female', 50, 'Amazon', 13, 'Systems Administrator', True)]

## Columns and their types

In [11]:
con.execute("PRAGMA table_info('employment_claude_sonnet_4_1000rows')").fetchdf()  

,cid,name,type,notnull,dflt_value,pk
0,0,Name,VARCHAR,False,None,False
1,1,Gender,VARCHAR,False,None,False
2,2,Age,BIGINT,False,None,False
3,3,Company,VARCHAR,False,None,False
4,4,Years of Experience,BIGINT,False,None,False
5,5,Job Applied For,VARCHAR,False,None,False
6,6,Hired,BOOLEAN,False,None,False


## employment_claude_sonnet_4_1000rows table analysis using Pandas-DuckDB

In [12]:
df_claude = con.execute("SELECT * FROM employment_claude_sonnet_4_1000rows").fetchdf()

In [13]:
print(df_claude.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Name                 1000 non-null   object
 1   Gender               1000 non-null   object
 2   Age                  1000 non-null   int64 
 3   Company              1000 non-null   object
 4   Years of Experience  1000 non-null   int64 
 5   Job Applied For      1000 non-null   object
 6   Hired                1000 non-null   bool  
dtypes: bool(1), int64(2), object(4)
memory usage: 48.0+ KB
None


In [14]:
print(df_claude.isnull().sum())

Name                   0
Gender                 0
Age                    0
Company                0
Years of Experience    0
Job Applied For        0
Hired                  0
dtype: int64


In [15]:
print(df_claude.nunique())

Name                   938
Gender                   3
Age                     44
Company                 94
Years of Experience     26
Job Applied For         81
Hired                    2
dtype: int64


In [16]:
print(df_claude.describe())

               Age  Years of Experience
count  1000.000000          1000.000000
mean     43.666000             8.774000
std      12.722827             7.130622
min      22.000000             0.000000
25%      33.000000             3.000000
50%      44.000000             7.000000
75%      55.000000            14.000000
max      65.000000            25.000000


## employment_gpt4o_1000rows table analysis using Pandas-DuckDB

In [17]:
df_gpt4o = con.execute("SELECT * FROM employment_gpt4o_1000rows").fetchdf()
print(df_gpt4o.isnull().sum())

Name                   0
Gender                 0
Age                    0
Company                0
Years of Experience    0
Job Applied For        0
Hired                  0
dtype: int64


In [18]:
# logical constraints as just an example looking if the person is adult and over 18 years old when working in a company
con.execute("""
    SELECT
        100.0 * SUM(CASE WHEN age >= 18 THEN 1 ELSE 0 END) / COUNT(*) AS valid_age,
    FROM employment_gpt4o_1000rows
""").fetchdf()

,valid_age
0,100.0
